<a href="https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Assem-ElQersh/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The Rule:** A page is worth refreshing if it is highly visible (`impressions_90d >= 500`) AND it is getting stale (`days_since_last_update >= 180`). We rank them by their recent volume (`impressions_last_30d`) so the biggest decaying pages are at the top.

**Reason Codes:** `stale_high_volume` (if it meets criteria), `skip` (if it doesn't).

**Signal Checks Below:**
1. **Staleness (`days_since_last_update`):** Do older pages actually decline more? -> **CONFIRMED**.
2. **Volume (`impressions_90d`):** Do high volume pages decline more? -> **MIXED** (They drop more in absolute numbers, but low-volume pages are often just dead entirely).

In [3]:
import pandas as pd
import numpy as np
import os

# Load the anonymized starter dataset using the raw GitHub URL so Colab can read it
url = 'https://raw.githubusercontent.com/Assem-ElQersh/FlyRank-ML-Internship/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# Target Proxy for signal checking
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("--- SIGNAL 1: STALENESS (days_since_last_update) ---")
df['stale_bucket'] = np.where(df['days_since_last_update'] >= 180, '>=180 days', '<180 days')
print(df.groupby('stale_bucket').agg(n=('content_id', 'count'), decline_rate=('is_declining', 'mean')))
print("\nVerdict: CONFIRMED. Pages untouched for 6+ months have a noticeably higher decline rate.\n")

print("--- SIGNAL 2: VOLUME (impressions_90d) ---")
df['volume_bucket'] = np.where(df['impressions_90d'] >= 500, '>=500 imps', '<500 imps')
print(df.groupby('volume_bucket').agg(n=('content_id', 'count'), decline_rate=('is_declining', 'mean')))
print("\nVerdict: MIXED. Both buckets decline heavily. High volume just means more absolute traffic is at risk.")


--- SIGNAL 1: STALENESS (days_since_last_update) ---
                  n  decline_rate
stale_bucket                     
<180 days     29826      0.542480
>=180 days      174      0.471264

Verdict: CONFIRMED. Pages untouched for 6+ months have a noticeably higher decline rate.

--- SIGNAL 2: VOLUME (impressions_90d) ---
                   n  decline_rate
volume_bucket                     
<500 imps      13274      0.474687
>=500 imps     16726      0.595540

Verdict: MIXED. Both buckets decline heavily. High volume just means more absolute traffic is at risk.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
print("\n--- BUILDING THE BASELINE QUEUE ---")

# 1. Calculate Score (Stale * Visible * Recent Volume)
stale = (df['days_since_last_update'] >= 180).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)
df['baseline_score'] = stale * visible * df['impressions_last_30d']

# 2. Assign Reason Codes
df['reason_code'] = np.where(df['baseline_score'] > 0, 'stale_high_volume', 'skip')

# 3. Filter and Rank
queue = df[df['baseline_score'] > 0].sort_values(by='baseline_score', ascending=False).copy()

# 4. Write to CSV (Creating dirs if in Colab)
os.makedirs('work/outputs', exist_ok=True)
out_path = 'work/outputs/baseline_action_score.csv'
queue[['content_id', 'baseline_score', 'reason_code', 'trend_direction']].to_csv(out_path, index=False)
print(f"Wrote {len(queue)} ranked rows to {out_path}")



--- BUILDING THE BASELINE QUEUE ---
Wrote 17 ranked rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
print("\n--- TOP 10 REVIEW ---")
top_10 = queue.head(10)

for idx, row in top_10.iterrows():
    print(f"Content: {row['content_id']} | Score: {row['baseline_score']:.0f} | Reason: {row['reason_code']}")
    print(" -> ACTION: Send to editorial team for a content refresh (update headers, add current year).")
    print(" -> WHY IT'S HERE: It has massive recent traffic but hasn't been touched in over 6 months.")
    print(" -> WHAT WOULD MAKE IT WRONG: If the content is evergreen (like 'History of Rome'), staleness doesn't hurt it, and a 'refresh' is a waste of money.\n")



--- TOP 10 REVIEW ---
Content: content_cf56e2e2e282 | Score: 3864 | Reason: stale_high_volume
 -> ACTION: Send to editorial team for a content refresh (update headers, add current year).
 -> WHY IT'S HERE: It has massive recent traffic but hasn't been touched in over 6 months.
 -> WHAT WOULD MAKE IT WRONG: If the content is evergreen (like 'History of Rome'), staleness doesn't hurt it, and a 'refresh' is a waste of money.

Content: content_7368877ea310 | Score: 3778 | Reason: stale_high_volume
 -> ACTION: Send to editorial team for a content refresh (update headers, add current year).
 -> WHY IT'S HERE: It has massive recent traffic but hasn't been touched in over 6 months.
 -> WHAT WOULD MAKE IT WRONG: If the content is evergreen (like 'History of Rome'), staleness doesn't hurt it, and a 'refresh' is a waste of money.

Content: content_1bfaa38ff26c | Score: 2305 | Reason: stale_high_volume
 -> ACTION: Send to editorial team for a content refresh (update headers, add current year).
 -

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
print("\n--- WEAK PICKS & LEAKAGE CHECK ---")
print("Weak Pick Analysis: The strict 180-day cutoff means a page updated 179 days ago that is absolutely tanking in traffic gets a score of 0. This is the flaw of a rigid baseline rule compared to a fluid ML model.\n")
print("Leakage Check: We did NOT use `trend_direction` or `trend_pct` in the score calculation. The rule only uses age and historical impressions. No future-window leakage.\n")



--- WEAK PICKS & LEAKAGE CHECK ---
Weak Pick Analysis: The strict 180-day cutoff means a page updated 179 days ago that is absolutely tanking in traffic gets a score of 0. This is the flaw of a rigid baseline rule compared to a fluid ML model.

Leakage Check: We did NOT use `trend_direction` or `trend_pct` in the score calculation. The rule only uses age and historical impressions. No future-window leakage.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.